## Open government data provided by **opendata.swiss**

_Automatically generated Python starter code for dataset_ **2be419d8-0dc5-4881-8e93-963d5a5cc772**


## Dataset

# Chômage, dès 2004


## Description

Ce jeu de données présente les principaux indicateurs mensuels du chômage dans le canton de Fribourg et pour l&#x27;ensemble de la Suisse, depuis 2004. Les données sont issues des statistiques du marché du travail du Secrétariat d&#x27;État à l&#x27;économie &#40;SECO&#41;.Demandeurs d&#x27;emploi inscrits : tous les demandeurs d&#x27;emploi, chômeurs et non-chômeurs, qui sont inscrits auprès d&#x27;un office régional de placement &#40;ORP&#41; et cherchent un emploi.Chômeurs : personnes annoncées auprès d&#x27;un ORP, qui n&#x27;ont pas d&#x27;emploi et sont immédiatement disponibles en vue d&#x27;un placement &#40;peu importe qu&#x27;elles touchent ou non une indemnité de chômage&#41;. Les chômeurs complets cherchent un poste à plein temps; les chômeurs partiels un poste à temps partiel.Les taux de chômage et de demandeurs d&#x27;emploi sont exprimés en % de la population active &#40;y compris les diplomates et les fonctionnaires internationaux résidant en Suisse&#41;. Le nombre de personnes actives est calculé sur la base:du Recensement fédéral de la population 2000 pour la période de 2004 à 2009 &#40;127 250 personnes actives pour le canton de Fribourg&#41;du Relevé structurel 2010 pour la période de 2010 à 2013 &#40;153 019 personnes actives, estimation à partir d&#x27;un relevé par échantillon&#41;du pooling du Relevé structurel 2012-2014 pour la période de 2014 à 2016 &#40;164 293 personnes actives, estimation à partir d&#x27;un relevé par échantillon&#41; du pooling du Relevé structurel 2015-2017 pour la période de 2017 à 2019 &#40;173 228 personnes actives, estimation à partir d&#x27;un relevé par échantillon&#41; du pooling du Relevé structurel 2018-2020 pour la période de 2020 à 2022 &#40;173 367 personnes actives, estimation à partir d&#x27;un relevé par échantillon&#41; du pooling du Relevé structurel 2021-2023 à partir de 2023 &#40;181 109 personnes actives, estimation à partir d&#x27;un relevé par échantillon&#41;. Les taux à partir de 2023 déjà publiés ont été révisés en juillet 2025.Pour plus d&#x27;informations : &#91;https://www.amstat.ch/v2/amstat&#95;fr.html&#93;&#40;https://www.amstat.ch/v2/amstat&#95;fr.html&#41;


## Dataset links

[View this dataset in the data catalogue](https://opendata.swiss/de/dataset/arbeitslosigkeit-ab-2004)


## Metadata

- **identifier:** 03&#95;03&#95;chomage&#95;mensuel@ssd
- **publisher.name:** ssd
- **organization.display&#95;name:** Amt für Statistik und Daten Freiburg
- **organization.url:** https://www.fr.ch/deef/ssd
- **maintainer:** 
- **maintainer&#95;email:** 
- **keywords:** demandeurs-demploi, chomeurs, taux-de-chomage, taux-de-demandeurs-demploi, conjoncture-fribourgeoise, indicateurs-conjoncturels
- **issued:** 2026-11-05 00:00:00
- **metadata&#95;created:** 2025-01-30T19:23:25.910949
- **metadata&#95;modified:** 2026-11-06T02:03:04.389927
- **Resource id:** 270d2c29-bbc5-4845-8949-f04116166f1d
- **Resource name:** csv
- **Resource package&#95;id:** 2be419d8-0dc5-4881-8e93-963d5a5cc772
- **Resource description:** Chômage, dès 2004 &#40;csv&#41;
- **Resource issued:** 2026-11-05 00:00:00
- **Resource modified:** 2026-11-05 08:48:54.223000+00:00
- **Resource license:** https://opendata.swiss/terms-of-use&#35;terms&#95;by
- **Resource rights:** http://www.opendefinition.org/licenses/cc-by-sa


## Imports and helper functions

This template targets pandas (3.x), seaborn (0.13+), matplotlib (3.x), and requests (2.x); packages are not installed or pinned by the notebook. Check the versions below when sharing results.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
print(f"pandas {pd.__version__}; seaborn {sns.__version__}")

In [ ]:
from io import BufferedReader
from typing import BinaryIO

import requests

USER_AGENT = "OpendataSwiss-StarterCode-Python/1.0"
HTTP_TIMEOUT = 30


def get_dataset(
    url: str | BinaryIO,
    *,
    sep: str | None = None,
    drop_empty_unnamed: bool = True,
    **kwargs: object,
) -> pd.DataFrame:
    """Read one CSV resource; preserve data and report parsing failures."""
    if isinstance(url, str) and url.startswith(("https://", "http://")):
        with requests.get(
            url, headers={"User-Agent": USER_AGENT}, timeout=HTTP_TIMEOUT, stream=True
        ) as response:
            response.raise_for_status()
            # Decode HTTP compression while streaming CSV bytes into pandas.
            response.raw.decode_content = True
            # Let the buffer finish reading after the HTTP connection reaches EOF.
            response.raw.auto_close = False
            with BufferedReader(response.raw) as source:
                # Detect file compression after HTTP compression has been decoded.
                if kwargs.get("compression", "infer") == "infer" and source.peek(2).startswith(
                    b"\x1f\x8b"
                ):
                    kwargs["compression"] = "gzip"
                return get_dataset(source, sep=sep, drop_empty_unnamed=drop_empty_unnamed, **kwargs)
    data = pd.read_csv(url, sep=sep, engine="python", dtype_backend="numpy_nullable", **kwargs)
    # Extra fields can make pandas silently consume leading values as an index.
    if kwargs.get("index_col") is None and not isinstance(data.index, pd.RangeIndex):
        raise pd.errors.ParserError(
            "CSV rows contain unexpected index fields. Check the delimiter and header; "
            "set index_col explicitly only if these fields are an intentional index."
        )
    if drop_empty_unnamed and len(data) > 0:
        # Blank CSV headers (often from trailing separators) become "Unnamed: n".
        unnamed = data.columns.astype(str).str.fullmatch(r"Unnamed: \d+(?:\.\d+)?")
        empty_unnamed = unnamed & data.isna().all()
        if empty_unnamed.any():
            print("Removed empty unnamed columns:", data.columns[empty_unnamed].tolist())
            data = data.loc[:, ~empty_unnamed]
    return data

## Load data

Selected resource: **csv** (CSV). This notebook loads only this resource.

For CSV, the separator is guessed. If parsing looks wrong (especially for a single-column file), set `sep=","` or `sep=";"` in `get_dataset(...)`. Check inferred types, missing-value codes, decimal marks, and identifiers with leading zeros; use `dtype`, `na_values`, `decimal`, or `encoding` as needed.

Unexpected index inference raises a parsing error to prevent values shifting under the wrong headers. Check the delimiter and header, or set `index_col` explicitly for an intentional index.

Fully empty columns named `Unnamed: n` are removed on import. Named empty columns and unnamed columns containing values are kept. Use `drop_empty_unnamed=False` to retain every column, including intentionally named `Unnamed: n` fields.


In [ ]:
df = get_dataset("https://opendata.fr.ch/api/explore/v2.1/catalog/datasets/03_03_chomage_mensuel/exports/csv?use_labels=true")

## Analyze data

Inspect before cleaning: these cells leave `df` unchanged. Duplicate rows and missing values may be meaningful; check the metadata before removing or filling them.


In [ ]:
print(f"{len(df):,} rows × {len(df.columns):,} columns")
print(f"{df.duplicated().sum():,} duplicate rows after the first occurrence")
df.info()
display(df.head())

In [ ]:
# Column types, completeness, and distinct non-missing values.
profile = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100),
        "unique": df.nunique(dropna=True),
    }
).sort_values("missing_pct", ascending=False)
display(profile.round(1))

In [ ]:
# Show up to 20 columns with missing values; percentages use all rows.
missing = profile.loc[profile["missing"] > 0].head(20).rename_axis("column").reset_index()
if not missing.empty:
    fig, ax = plt.subplots(figsize=(8, max(2, len(missing) * 0.3)))
    sns.barplot(data=missing, x="missing_pct", y="column", color="steelblue", errorbar=None, ax=ax)
    ax.set(xlim=(0, 100), xlabel="Missing values (%)", ylabel="", title="Missingness by column")
    fig.tight_layout()
    plt.show()
else:
    print("No missing values to plot (or the dataset has no rows).")

In [ ]:
# Summarize numeric and non-numeric columns, including pandas string types.
if not df.empty:
    display(df.describe(include="all").T)
else:
    print("No data to summarize.")

In [ ]:
# Plot up to four numeric columns. Exclude missing/infinite values only from plots.
numeric = df.select_dtypes(include="number")
for column in numeric.columns[:4]:
    values = numeric[column].dropna()
    values = values.loc[~values.isin([float("inf"), -float("inf")])]
    if not values.empty:
        fig, ax = plt.subplots(figsize=(7, 3))
        sns.histplot(x=values, bins=30, ax=ax)
        ax.set(title=column, xlabel=column, ylabel="Count")
        fig.tight_layout()
        plt.show()

In [ ]:
# Top 10 values for up to three categorical columns, including missing values.
categorical = df.select_dtypes(include=["string", "object", "category", "bool"])
for column in categorical.columns[:3]:
    print(column)
    display(df[column].value_counts(dropna=False).head(10).to_frame("count"))

### Next steps

Choose columns from `df.columns`: compare groups, inspect a time trend, or plot two numeric variables. Adapt the commented examples below; use the metadata to decide which comparisons make sense.


In [ ]:
# Group summary (replace column names):
# df.groupby("group", dropna=False)["value"].agg(["count", "mean", "median"])

# Parse a date explicitly, then inspect a time trend:
# dated = df.assign(date=pd.to_datetime(df["date"], format="%Y-%m-%d", errors="raise"))
# dated.groupby("date")["value"].mean().plot(ylabel="Mean value")

# Relationship between two numeric variables:
# sns.scatterplot(data=df, x="x_column", y="y_column", alpha=0.5)

**Questions about the data?** opendata@fr.ch &#124; Centre de compétences et de coordination OGD / Kompetenz- und Koordinationszentrum für OGD
